# Pepper Impurity Detection

## 
<ol>
    <li>Data loading</li>
    <li>Data Preprocessing</li>
    <li>Data Augmentation</li>
    <li>Train the Model</li>
    <li>Test the Model</li>
    <li>Evalute the Model</li>
</ol>


In [ ]:
import os
from PIL import Image, ImageFilter, ImageEnhance
import cv2
import numpy as np
import pillow_heif

## Step 01 : Data Loading

In [ ]:
# Input direct loading
input_dir = "..\\dataset\\Pepper Seed Impurity Research 1.v1i.yolov8\\train\\images"

# Output direct saving
output_dir = "..\\dataset\\Pepper Seed Impurity Research 1.v1i.yolov8\\train\\results"

# Check if the image was loaded successfully
if not os.path.exists(input_dir):
    raise FileNotFoundError(f"Directory not found: {input_dir}")


# Get the file count in the input directory
file_count = len([name for name in os.listdir(input_dir) if os.path.isfile(os.path.join(input_dir, name))])
print("File count in the input directory: ", file_count)

## Step 02 : Image Preprocessing

In [ ]:
def preprocess_image_impurity_aware(input_dir, output_dir):
    """
    Enhanced preprocessing optimized for detecting small impurities
    alongside black pepper seeds
    """
    # Supported image extensions
    img_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.heic')
    
    # Create the output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    for filename in os.listdir(input_dir):
        if not filename.lower().endswith(img_extensions):
            continue
            
        try:
            image_path = os.path.join(input_dir, filename)
            output_path = os.path.join(output_dir, filename)
            
            # Handle HEIC conversion
            if filename.lower().endswith('.heic'):
                heif_file = pillow_heif.read(image_path)
                image = Image.frombytes(
                    heif_file.mode,
                    heif_file.size,
                    heif_file.data,
                    "raw",
                    heif_file.mode,
                    heif_file.stride,
                )
                output_path = output_path[:-5] + '.jpg'
            else:
                image = Image.open(image_path)
            
            # Keep original size - important for small objects
            # DON'T resize to 416x416 - use at least 640x640 if needed
            
            # DON'T convert to grayscale - color helps distinguish impurities
            
            # Enhance contrast conservatively for better feature visibility
            enhancer = ImageEnhance.Contrast(image)
            image = enhancer.enhance(1.2)  # Reduced from 1.3 to preserve details
            
            # Mild sharpening to enhance edges of small objects
            image = image.filter(ImageFilter.SHARPEN)
            
            # More gentle contrast adjustment
            image_array = np.array(image)
            image_array = cv2.convertScaleAbs(image_array, alpha=1.3, beta=5)
            image = Image.fromarray(image_array)
            
            # Save the processed image
            image.save(output_path)
            
            print(f"Processed: {filename}")
            
        except Exception as e:
            print(f"Error processing {filename}: {str(e)}")
            continue
            
    print(f"Preprocessing complete. Output saved to: {output_dir}")

In [ ]:
preprocess_image_impurity_aware(input_dir, output_dir)

In [58]:
# Check if the image was loaded successfully
if output_dir is None:
    raise FileNotFoundError(f"Image not found at path: {output_dir}")

# Get the file count in the output directory
file_count = len([name for name in os.listdir(output_dir) if os.path.isfile(os.path.join(output_dir, name))])
print("File count in the output directory: ", file_count)

File count in the output directory:  966


## Step 03 : Data Augmentaion

In [ ]:
def augment_image(input_dir, output_dir):
    # Supported image extensions
    img_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.heic')
    
    # Create the output directory as augmented 
    os.makedirs(output_dir, exist_ok=True)
    
    for filename in os.listdir(input_dir):
        if not filename.lower().endswith(img_extensions):
            continue
            
        try:
            image_path = os.path.join(input_dir, filename)
            output_path = os.path.join(output_dir, filename)
            
            # Handle HEIC conversion
            if filename.lower().endswith('.heic'):
                heif_file = pillow_heif.read(image_path)
                image = Image.frombytes(
                    heif_file.mode,
                    heif_file.size,
                    heif_file.data,
                    "raw",
                    heif_file.mode,
                    heif_file.stride,
                )
                output_path = output_path[:-5] + '.jpg'
            else:
                image = Image.open(image_path)
            
            # Blur
            image_blur = image.filter(ImageFilter.GaussianBlur(radius=2))
            image_blur.save(output_path[:-4] + "_blur.jpg")
            
            # Rotation
            image_rotated = image.rotate(45)
            image_rotated.save(output_path[:-4] + "_rotated.jpg")
            
            # Hue
            image_hue = ImageEnhance.Color(image).enhance(2)
            image_hue.save(output_path[:-4] + "_hue.jpg")
            
            # Saturation
            image_saturation = ImageEnhance.Color(image).enhance(2)
            image_saturation.save(output_path[:-4] + "_saturation.jpg")
            
            # Brightness
            image_brightness = ImageEnhance.Brightness(image).enhance(2)
            image_brightness.save(output_path[:-4] + "_brightness.jpg")
            
            # Exposure
            image_exposure = ImageEnhance.Contrast(image).enhance(2)
            image_exposure.save(output_path[:-4] + "_exposure.jpg")
            
            # Noise
            image_array = np.array(image)
            image_array = cv2.GaussianBlur(image_array, (5, 5), 0)
            image_noise = Image.fromarray(image_array)
            image_noise.save(output_path[:-4] + "_noise.jpg")
            
            print(f"Augmented: {filename}")
            
        except Exception as e:
            print(f"Error augmenting {filename}: {str(e)}")
            continue

    print(f"Augmentation complete. Output saved to: {output_dir}")

In [ ]:
# Output direct saving
input_dir_aug = "..\\dataset\\Pepper Seed Impurity Research 1.v1i.yolov8\\train\\results"
output_dir_aug = "../dataset/Pepper Seed Impurity Research 1.v1i.yolov8/train/augmented"

# Augment the images
augment_image(input_dir_aug, output_dir_aug)

In [43]:
# Check if the image was loaded successfully
if output_dir_aug is None:
    raise FileNotFoundError(f"Image not found at path: {output_dir_aug}")

# Get the file count in the output directory
file_count = len([name for name in os.listdir(output_dir_aug) if os.path.isfile(os.path.join(output_dir_aug, name))])
print("File count in the output directory: ", file_count)

File count in the output directory:  6762


## Step 04 : Yolo model v8 train

Object detection by YOLOv8
1. Load the YOLOv8 model
2. Load the image
3. Train the model



In [59]:
# Load the YOLOv8 model
from ultralytics import YOLO

In [ ]:
# Load the YOLOv8 model
model = YOLO("yolov8s.pt")

In [ ]:
# Load the image
data_path = "C:\\Users\\PC\\OneDrive\\Desktop\\Pepper_Research\\dataset\\Pepper Seed Impurity Research 1.v1i.yolov8\\data.yaml"

In [ ]:
# Model training configuration optimized for small impurity detection
results = model.train(
    data=data_path,                         # Path to your dataset YAML
    epochs=200,                             # Increased for better learning
    imgsz=640,                              # 640px works well for small objects
    batch=8,                                # Adjusted for memory constraints
    device=0,                               # Use GPU
    workers=2,                              # Data loading threads
    
    # Small object detection improvements
    min_items=1,                            # Allow training with just 1 object in an image
    overlap_mask=True,                      # Better segmentation for overlapping objects
    
    # Training optimization
    optimizer='AdamW',                      # Modern optimizer for better convergence
    lr0=0.001,                              # Lower initial learning rate for stability
    lrf=0.01,                               # Final learning rate
    cos_lr=True,                            # Cosine learning rate schedule
    weight_decay=0.0005,                    # Weight decay to prevent overfitting
    warmup_epochs=3,                        # Learning rate warmup
    warmup_momentum=0.8,                    # Warmup momentum
    
    # Class balance (critical for impurity detection)
    class_weights=[1.0, 1.5],               # Weight impurities 1.5x more than pepper
    
    # Effective augmentation for small objects
    augment=True,                           # Enable built-in augmentation
    mosaic=0.8,                             # Mosaic augmentation (good for small objects)
    mixup=0.1,                              # Light mixup for variety
    hsv_h=0.015,                            # Hue variation
    hsv_s=0.3,                              # Saturation variation
    hsv_v=0.3,                              # Value variation
    degrees=10.0,                           # Rotation range
    translate=0.1,                          # Translation
    scale=0.5,                              # Scaling
    shear=2.0,                              # Shear
    fliplr=0.5,                             # Horizontal flip probability
    
    # Training stability
    patience=50,                            # Early stopping patience
    save_period=10,                         # Save checkpoint every 10 epochs
    cache=False,                            # Disable cache to reduce memory
    single_cls=False,                       # Multi-class detection
    verbose=True,                           # Show details
)

In [65]:
# Load your trained model
model = YOLO('runs/detect/train19/weights/best.pt')  

In [68]:
# Predict on new images
results = model.predict(
    source='..\\dataset\\Pepper Seed Impurity Research 1.v1i.yolov8\\test\\images\\BlackPepper-289-_jpg.rf.52bb8645588f58377b48f12c3b5c112e.jpg',  # or folder/video
    conf=0.5,                # confidence threshold
    save=True                # saves to runs/detect/predict
)


image 1/1 c:\Users\PC\OneDrive\Desktop\Pepper_Research\notebooks\..\dataset\Pepper Seed Impurity Research 1.v1i.yolov8\test\images\BlackPepper-289-_jpg.rf.52bb8645588f58377b48f12c3b5c112e.jpg: 640x640 283 Black-pepper-seeds, 58.3ms
Speed: 3.6ms preprocess, 58.3ms inference, 5.5ms postprocess per image at shape (1, 3, 640, 640)
Results saved to runs\detect\predict


In [ ]:
def calculate_purity(detection_results, area_based=True):
    """
    Calculate purity percentage based on either count or area
    
    Args:
        detection_results: Results from YOLO prediction
        area_based: If True, calculate purity based on area rather than count
    
    Returns:
        dict: Contains purity percentage and counts/areas
    """
    # Initialize counters
    black_pepper_count = 0
    impurity_count = 0
    black_pepper_area = 0
    impurity_area = 0
    
    # Process each detection
    for result in detection_results:
        boxes = result.boxes
        for box in boxes:
            cls = int(box.cls[0])
            conf = float(box.conf[0])
            
            # Calculate area of detection
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            area = (x2 - x1) * (y2 - y1)
            
            # Count objects and areas
            if cls == 0:  # Black-pepper-seed
                black_pepper_count += 1
                black_pepper_area += area
            else:  # Impurity
                impurity_count += 1
                impurity_area += area
    
    # Calculate percentages
    total_count = black_pepper_count + impurity_count
    total_area = black_pepper_area + impurity_area
    
    if total_count == 0:
        return {"error": "No objects detected"}
    
    # Calculate based on preferred method
    if area_based and total_area > 0:
        purity_percentage = (black_pepper_area / total_area) * 100
        impurity_percentage = (impurity_area / total_area) * 100
    else:
        purity_percentage = (black_pepper_count / total_count) * 100
        impurity_percentage = (impurity_count / total_count) * 100
    
    return {
        "purity_percentage": purity_percentage,
        "impurity_percentage": impurity_percentage,
        "black_pepper_count": black_pepper_count,
        "impurity_count": impurity_count,
        "total_count": total_count,
        "black_pepper_area": black_pepper_area,
        "impurity_area": impurity_area,
        "area_based_calculation": area_based
    }

In [ ]:
def detailed_purity_analysis(image_path, model, conf_threshold=0.5, 
                            area_based=True, save_visualization=True,
                            output_dir="purity_reports"):
    """
    Production-ready purity analysis with detailed report and visualization
    
    Args:
        image_path: Image to analyze
        model: Trained YOLO model
        conf_threshold: Detection confidence threshold
        area_based: Use area-based (True) or count-based (False) calculation
        save_visualization: Whether to save annotated image
        output_dir: Directory to save reports
        
    Returns:
        dict: Comprehensive purity analysis
    """
    os.makedirs(output_dir, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Run prediction with verbose=False to minimize console output
    results = model.predict(
        source=image_path,
        conf=conf_threshold,
        save=save_visualization,
        save_txt=True,
        save_conf=True,
        verbose=False
    )
    
    # Get base purity stats
    purity_stats = calculate_purity(results, area_based=area_based)
    
    # Add confidence analysis
    confidence_data = []
    size_data = {"pepper": [], "impurity": []}
    
    # Analyze detections in detail
    for result in results:
        boxes = result.boxes
        img_shape = result.orig_shape  # (height, width)
        img_area = img_shape[0] * img_shape[1]
        
        for box in boxes:
            cls = int(box.cls[0])
            conf = float(box.conf[0])
            
            # Get bounding box details
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            area = (x2 - x1) * (y2 - y1)
            area_percent = (area / img_area) * 100
            
            # Store confidence and size data
            confidence_data.append({"class": cls, "confidence": conf})
            
            if cls == 0:  # Pepper
                size_data["pepper"].append({"area_px": area, "area_percent": area_percent})
            else:  # Impurity
                size_data["impurity"].append({"area_px": area, "area_percent": area_percent})
    
    # Calculate confidence statistics
    if confidence_data:
        pepper_conf = [d["confidence"] for d in confidence_data if d["class"] == 0]
        impurity_conf = [d["confidence"] for d in confidence_data if d["class"] == 1]
        
        confidence_stats = {
            "pepper_confidence": {
                "mean": np.mean(pepper_conf) if pepper_conf else 0,
                "min": min(pepper_conf) if pepper_conf else 0,
                "max": max(pepper_conf) if pepper_conf else 0
            },
            "impurity_confidence": {
                "mean": np.mean(impurity_conf) if impurity_conf else 0,
                "min": min(impurity_conf) if impurity_conf else 0,
                "max": max(impurity_conf) if impurity_conf else 0
            }
        }
    else:
        confidence_stats = {"error": "No detections to analyze"}
    
    # Combine all information
    full_analysis = {
        **purity_stats,
        "confidence_stats": confidence_stats,
        "size_data": size_data,
        "analysis_parameters": {
            "confidence_threshold": conf_threshold,
            "area_based_calculation": area_based,
            "image_analyzed": os.path.basename(image_path)
        },
        "timestamp": timestamp
    }
    
    # Save report as JSON
    report_file = os.path.join(output_dir, f"purity_report_{timestamp}.json")
    with open(report_file, 'w') as f:
        json.dump(full_analysis, f, indent=4)
    
    # Print summary report
    print("\n===== DETAILED PURITY REPORT =====")
    print(f"Image: {os.path.basename(image_path)}")
    print(f"Analysis method: {'Area-based' if area_based else 'Count-based'}")
    print(f"Purity: {purity_stats['purity_percentage']:.2f}%")
    print(f"Impurity: {purity_stats['impurity_percentage']:.2f}%")
    print(f"\nDetected objects:")
    print(f"- Black Pepper Seeds: {purity_stats['black_pepper_count']} seeds")
    print(f"- Impurities: {purity_stats['impurity_count']} items")
    print(f"- Total Objects: {purity_stats['total_count']} items")
    
    if 'black_pepper_area' in purity_stats:
        print(f"\nArea coverage:")
        print(f"- Black Pepper Area: {purity_stats['black_pepper_area']:.2f} px²")
        print(f"- Impurity Area: {purity_stats['impurity_area']:.2f} px²")
    
    print(f"\nReport saved to: {report_file}")
    
    return full_analysis

In [ ]:
# Test with a single image
single_image_path = '..\\dataset\\Pepper Seed Impurity Research 1.v1i.yolov8\\test\\images\\BlackPepper-289-_jpg.rf.52bb8645588f58377b48f12c3b5c112e.jpg'
analyze_image_purity(single_image_path, model)

# To test with a directory of images (uncomment to use)
# test_directory = '..\\dataset\\Pepper Seed Impurity Research 1.v1i.yolov8\\test\\images'
# analyze_image_purity(test_directory, model)

In [ ]:
def comprehensive_model_validation(model, test_data_dir):
    """
    Comprehensive model validation with detailed metrics for pepper purity detection
    
    Args:
        model: Trained YOLO model
        test_data_dir: Directory with test images
        
    Returns:
        dict: Validation results
    """
    print("\n===== COMPREHENSIVE MODEL VALIDATION =====")
    
    # 1. Standard validation metrics on test set
    print("\nRunning standard validation...")
    metrics = model.val(data=data_path, split='test')
    
    # 2. Calculate confusion matrix
    print("\nGenerating confusion matrix...")
    cm = model.val(data=data_path, split='test', conf=0.25, iou=0.6, save_json=True, verbose=False)
    
    # 3. Test different confidence thresholds
    print("\nEvaluating different confidence thresholds...")
    thresholds = [0.25, 0.5, 0.75]
    threshold_results = {}
    
    for conf in thresholds:
        print(f"\nTesting confidence threshold: {conf}")
        test_dir = os.path.join(os.path.dirname(data_path), "test/images")
        
        if os.path.exists(test_dir):
            # Sample just 5 images to avoid overwhelming output
            test_images = [os.path.join(test_dir, f) for f in os.listdir(test_dir) 
                          if f.endswith(('.jpg', '.jpeg', '.png'))][:5]
            
            conf_results = []
            for img in test_images:
                print(f"\nAnalyzing: {os.path.basename(img)} with conf={conf}")
                result = analyze_image_purity(img, model, conf_threshold=conf, save=False)
                conf_results.append(result)
            
            threshold_results[conf] = conf_results
    
    # 4. Test on challenging samples (low light, crowded, etc)
    print("\nTesting on challenging samples...")
    challenging_dir = os.path.join(os.path.dirname(data_path), "challenging")
    if os.path.exists(challenging_dir):
        for img in os.listdir(challenging_dir):
            if img.endswith(('.jpg', '.jpeg', '.png')):
                print(f"\nAnalyzing challenging image: {img}")
                img_path = os.path.join(challenging_dir, img)
                analyze_image_purity(img_path, model)
    
    # 5. Summarize findings
    print("\n===== VALIDATION SUMMARY =====")
    print(f"mAP@50-95: {metrics.box.map}")
    print(f"mAP@50: {metrics.box.map50}")
    print(f"Precision: {metrics.box.p}")
    print(f"Recall: {metrics.box.r}")
    
    # Return all collected validation data
    return {
        "metrics": metrics,
        "threshold_tests": threshold_results
    }

In [ ]:
def impurity_sample_extractor(mixed_images_dir, output_dir, model, conf_threshold=0.4):
    """
    Extracts impurity patches from mixed samples to build an impurity dataset
    
    Args:
        mixed_images_dir: Directory with mixed pepper+impurity images
        output_dir: Directory to save extracted impurity patches
        model: YOLO model that can detect impurities
        conf_threshold: Confidence threshold for detection
    """
    # Create output directories
    pepper_dir = os.path.join(output_dir, "pepper_patches")
    impurity_dir = os.path.join(output_dir, "impurity_patches")
    os.makedirs(pepper_dir, exist_ok=True)
    os.makedirs(impurity_dir, exist_ok=True)
    
    # Process each image
    img_extensions = ('.jpg', '.jpeg', '.png', '.bmp')
    image_files = [f for f in os.listdir(mixed_images_dir) 
                  if f.lower().endswith(img_extensions)]
    
    print(f"Processing {len(image_files)} images...")
    
    for idx, img_file in enumerate(image_files):
        img_path = os.path.join(mixed_images_dir, img_file)
        
        # Run detection
        results = model.predict(
            source=img_path, 
            conf=conf_threshold,
            save=False,
            verbose=False
        )
        
        # Extract patches
        try:
            # Load original image
            img = cv2.imread(img_path)
            if img is None:
                print(f"Error loading {img_file}, skipping...")
                continue
                
            # Process detections
            for result in results:
                boxes = result.boxes
                
                for i, box in enumerate(boxes):
                    cls = int(box.cls[0])
                    conf = float(box.conf[0])
                    
                    # Get bounding box with some margin
                    x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                    
                    # Add margin (20 pixels) to capture more context
                    margin = 20
                    h, w = img.shape[:2]
                    x1 = max(0, x1 - margin)
                    y1 = max(0, y1 - margin)
                    x2 = min(w, x2 + margin)
                    y2 = min(h, y2 + margin)
                    
                    # Extract patch
                    patch = img[y1:y2, x1:x2]
                    
                    # Skip if patch is too small
                    if patch.shape[0] < 20 or patch.shape[1] < 20:
                        continue
                    
                    # Save to appropriate directory
                    if cls == 0:  # Pepper
                        save_dir = pepper_dir
                        prefix = "pepper"
                    else:  # Impurity
                        save_dir = impurity_dir
                        prefix = "impurity"
                        
                    patch_filename = f"{prefix}_{os.path.splitext(img_file)[0]}_{i}_{conf:.2f}.jpg"
                    patch_path = os.path.join(save_dir, patch_filename)
                    
                    cv2.imwrite(patch_path, patch)
            
            if (idx + 1) % 10 == 0:
                print(f"Processed {idx + 1}/{len(image_files)} images...")
                
        except Exception as e:
            print(f"Error processing {img_file}: {str(e)}")
            continue
    
    # Count extracted patches
    pepper_count = len(os.listdir(pepper_dir))
    impurity_count = len(os.listdir(impurity_dir))
    
    print(f"\nExtraction complete:")
    print(f"- Extracted {pepper_count} pepper samples to {pepper_dir}")
    print(f"- Extracted {impurity_count} impurity samples to {impurity_dir}")
    
    return {
        "pepper_patches": pepper_count,
        "impurity_patches": impurity_count
    }

## Export for Development

In [ ]:
# Export to ONNX format (for production)
model.export(format='onnx')

In [ ]:
model.train(resume=True, epochs=50)  # Continue training